# 03 - StreetCLIP zero-shot and linear probe (Thea)

Models **5 and 6** of the results table. Needs a **GPU** (Kaggle / Colab).

1. **Zero-shot**: no training. Score = similarity between the image and the text *"A Street View photo in {country}."*
2. **Linear probe**: embed every image once (the slow step), save the embeddings for the team, train one small softmax layer on top.

Rules: same cropped images as everyone (`geosense.preprocess`), 336 px, all choices made on `val`, `test` only at the end.

**First do a dry run with `SUBSET = 1`** to check that everything works (a few minutes), then set `SUBSET = None`.

In [ ]:
import json, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from geosense import load_classes, load_manifest
from geosense.data import PROJECT_ROOT
from geosense.predictions import PRED_DIR, check_val, save_logits
from geosense.preprocess import find_dataset_root

# --- configuration ---------------------------------------------------------------------------
DATA_ROOT = find_dataset_root()      # folder that contains France/, Japan/, ...  (or pass the path)
SUBSET = None                        # dry run: e.g. 1 = one image per country and split. None = everything
# ---------------------------------------------------------------------------------------------
classes, class_weights = load_classes()
C = len(classes)
manifest = load_manifest()
if SUBSET:                           # quick test of the whole notebook; writes to predictions_dryrun/
    manifest = manifest.groupby(["split", "country"], group_keys=False).head(SUBSET).reset_index(drop=True)
OUT = PRED_DIR if SUBSET is None else PROJECT_ROOT / "predictions_dryrun"
part = {s: manifest[manifest.split == s].reset_index(drop=True) for s in ("train", "val", "test")}
y = {s: part[s].label.to_numpy() for s in part}
print(DATA_ROOT, "|", C, "classes |", {s: len(d) for s, d in part.items()}, "| output:", OUT)

import torch
from geosense.preprocess import ViewDataset
from geosense.streetclip import (TEMPLATE_SETS, class_weight_options, image_embeddings, load_streetclip,
                                 text_weights, train_probe, zero_shot_logits)
from geosense.baselines import macro_f1
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
assert device == 'cuda' or SUBSET, 'Use a GPU for the full run (Kaggle: Settings -> Accelerator)'

In [ ]:
model, tok = load_streetclip()          # ~1.7 GB download the first time
model = model.to(device)
scale = float(model.logit_scale.exp())
W0 = text_weights(model, tok, classes, TEMPLATE_SETS['street_view'], device)
print('text weights', W0.shape, '| scale', round(scale, 1))

## 1. How to turn the wide image into a square? (team rule: never squash)
Default = resize the short side, then centre-crop. Seif's EDA suggested squashing. Compare both **on a validation sample** with zero-shot macro-F1 (cheap) and use the same choice for the whole notebook. Tell the team which one you used.

In [ ]:
rng = np.random.default_rng(0)
idx = rng.choice(len(part['val']), size=min(1500, len(part['val'])), replace=False)
sample = part['val'].iloc[idx].reset_index(drop=True)
res = {}
for mode in ('centre', 'squash'):
    Zs = image_embeddings(model, ViewDataset(DATA_ROOT, sample.path, 336, mode), device, desc=mode)
    res[mode] = macro_f1(sample.label.to_numpy(), zero_shot_logits(Zs, W0, scale))
print({k: round(v, 3) for k, v in res.items()})
VIEW_MODE = max(res, key=res.get)
print('chosen:', VIEW_MODE)

## 2. Embed every image once (slow step)
**Time the first batches** (the progress bar shows it). Embeddings are cached in `predictions/streetclip_embeddings_<split>.npy` (unit-norm, saved as float16 to halve the size; load with `.astype('float32')`). On Kaggle the session can stop: the cache means you only lose the split in progress.

In [ ]:
OUT.mkdir(exist_ok=True)
Z = {}
for s in ('train', 'val', 'test'):
    f = OUT / f'streetclip_embeddings_{s}.npy'
    if f.exists():
        Z[s] = np.load(f).astype('float32')
    else:
        t = time.time()
        Z[s] = image_embeddings(model, ViewDataset(DATA_ROOT, part[s].path, 336, VIEW_MODE), device, desc=s)
        np.save(f, Z[s].astype('float16')); print(s, f'{(time.time()-t)/60:.1f} min')
print({s: v.shape for s, v in Z.items()})

## 3. Zero-shot: pick the prompt on validation


In [ ]:
best = (-1, None, None)
for name, templates in TEMPLATE_SETS.items():
    W = text_weights(model, tok, classes, templates, device)
    f1 = macro_f1(y['val'], zero_shot_logits(Z['val'], W, scale))
    print(f'{name:<12} val macro-F1 {f1:.3f}')
    if f1 > best[0]: best = (f1, name, W)
zs_f1, zs_name, W_zs = best
print('chosen prompt set:', zs_name, TEMPLATE_SETS[zs_name])
for s in ('val', 'test'):
    save_logits('streetclip_zeroshot', s, zero_shot_logits(Z[s], W_zs, scale), OUT, manifest, C)

## 4. Linear probe
One weight vector per country on the frozen embeddings. Two starts (from the text embeddings, or random) and three loss weightings (none, square-root, inverse frequency); the best combination on **val macro-F1** wins.

In [ ]:
opts = class_weight_options(np.array(class_weights))
results = {}
for init_name, init in (('text', W_zs), ('random', None)):
    for w_name, cw in opts.items():
        W, hist, f1 = train_probe(Z['train'], y['train'], Z['val'], y['val'], init=init, class_weights=cw, epochs=40,
                                  scale=scale, device=device, n_classes=C)
        results[(init_name, w_name)] = (f1, W, hist)
        print(f'init={init_name:<6} weights={w_name:<8} best val macro-F1 {f1:.3f} (epoch {int(np.argmax([h["val_macro_f1"] for h in hist])) + 1})')
(init_name, w_name), (probe_f1, W_probe, hist) = max(results.items(), key=lambda kv: kv[1][0])
print('chosen:', init_name, w_name)
for s in ('val', 'test'):
    save_logits('streetclip_probe', s, zero_shot_logits(Z[s], W_probe, scale), OUT, manifest, C)

In [ ]:
plt.plot([h['val_macro_f1'] for h in hist]); plt.axhline(zs_f1, ls='--', c='grey', label='zero-shot')
plt.xlabel('epoch'); plt.ylabel('val macro-F1'); plt.title('Linear probe'); plt.legend()
(PROJECT_ROOT / 'results').mkdir(exist_ok=True); plt.savefig(PROJECT_ROOT / 'results' / ('thea_probe_curve.png' if SUBSET is None else 'thea_probe_curve_dry.png'), dpi=150)

## 5. Check the exported files (validation only)

In [ ]:
check_val(OUT, manifest)

### Then
1. `git add predictions/*logits.npy` and push (small files). **Do not commit the embeddings** (50 MB+): put `streetclip_embeddings_*.npy` on Google Drive or a private Kaggle dataset and share the link.
2. Tell Louis (app, needs zero-shot), Faris (compare with fine-tuning) and Maya (evaluation) which `VIEW_MODE` and prompt set you used.

### Notes for the report
- Zero-shot idea (Lecture 16): aligned image-text embeddings, the text encoder builds the classifier.
- Pre-training overlap: StreetCLIP saw street imagery; say it can inflate scores.
- Licence CC-BY-NC: non-commercial only.